In [8]:
!git clone https://github.com/Frontman-11/Yoruba-to-english-translation.git

fatal: destination path 'Yoruba-to-english-translation' already exists and is not an empty directory.


In [9]:
!pip install -q --upgrade pip
# !pip install --upgrade --force-reinstall tensorflow tensorboard
!pip install -q sentencepiece

In [10]:
import sys

PATHS = ["/kaggle/working/Yoruba-to-english-translation"]
for PATH in PATHS:
    if PATH in sys.path:
        sys.path.remove(PATH)  
    sys.path.append(PATH)

import os
import datetime
import pandas as pd
import tensorflow as tf
from adaptive_softmax import AdaptiveSoftmax
from tokenizer_function import FrontmanTokenizer
from transformer_components import PositionalEncoding, DecoderTransformerBlock, EncoderTransformerBlock

In [11]:
print(f'Tensorflow version: {tf.__version__}')
print(f'Python version: {sys.version}')

Tensorflow version: 2.18.0
Python version: 3.10.16 (main, Feb  4 2025, 07:28:23) [GCC 12.2.0]


In [ ]:
tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
tf.tpu.experimental.initialize_tpu_system(tpu)
tpu_strategy = tf.distribute.TPUStrategy(tpu)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local


free(): corrupted unsorted chunks
https://symbolize.stripped_domain/r/?trace=7907fade4ebc,7907fad9604f,594591a4484f,594591a4484f&map= 
*** SIGABRT received by PID 10 (TID 1003) on cpu 31 from PID 10; stack trace: ***
PC: @     0x7907fade4ebc  (unknown)  (unknown)
    @     0x7906e4f02841       1888  (unknown)
    @     0x7907fad96050       9488  (unknown)
    @     0x594591a44850  (unknown)  (unknown)
    @     0x594591a44850  (unknown)  (unknown)
https://symbolize.stripped_domain/r/?trace=7907fade4ebc,7906e4f02840,7907fad9604f,594591a4484f,594591a4484f&map= 
E0301 13:18:34.905752    1003 coredump_hook.cc:316] RAW: Remote crash data gathering hook invoked.
E0301 13:18:34.905771    1003 client.cc:269] RAW: Coroner client retries enabled, will retry for up to 30 sec.
E0301 13:18:34.905776    1003 coredump_hook.cc:411] RAW: Sending fingerprint to remote end.
E0301 13:18:34.905800    1003 coredump_hook.cc:420] RAW: Cannot send fingerprint to Coroner: [NOT_FOUND] stat failed on crash report

In [ ]:
df = pd.read_csv('/kaggle/input/yoruba-english-pair/JW300_en-yo')
df.dropna(how='any', inplace=True)

In [ ]:
df.info()

# Global ${MAX\_SEQ\_LENGTH\ =\ 128}$

In [ ]:
MAX_SEQ_LENGTH = 128

In [ ]:
tokenizer = FrontmanTokenizer(
    model_path='/kaggle/input/tokenizer/yo_en_bpe.model',
    max_length=MAX_SEQ_LENGTH,
    truncation=True,
)

In [ ]:
def create_dataset(df, tokenizer, replica_batch_size=64, drop_remainder=True, shuffle_size=False, cache=False):
    encoder_input = tokenizer.encode(df['yoruba'].values.tolist(), with_attention_mask=True)
    decoder_input = tokenizer.encode(df['english'].values.tolist(), add_bos=True, with_attention_mask=True)
    decoder_target= tokenizer.encode(df['english'].values.tolist(), add_eos=True)
    
    dataset = tf.data.Dataset.from_tensor_slices((({
        "encoder_input_ids": encoder_input['input_ids'],
        "encoder_attention_mask": encoder_input['attention_mask'],
        "decoder_input_ids": decoder_input['input_ids'],
        "decoder_attention_mask": decoder_input['attention_mask'],}
        ),
        decoder_target
        )
    )

    if shuffle_size:
        dataset = dataset.shuffle(shuffle_size)
        
    if replica_batch_size:
        BATCH_SIZE = replica_batch_size * tpu_strategy.num_replicas_in_sync
        dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

    if cache:
        dataset = dataset.cache()
    return dataset.prefetch(tf.data.AUTOTUNE)



# def create_dataset(df, tokenizer, replica_batch_size=64, drop_remainder=True, shuffle_size=False, cache=False):
#     # Batch tokenize for efficiency
#     encoder_input = tokenizer.encode(df['yoruba'].tolist(), with_attention_mask=True)
#     decoder_input = tokenizer.encode(df['english'].tolist(), add_bos=True, with_attention_mask=True)
#     decoder_target = tokenizer.encode(df['english'].tolist(), add_eos=True)

#     def gen():
#         for enc_in, enc_mask, dec_in, dec_mask, dec_tgt in zip(
#             encoder_input['input_ids'], encoder_input['attention_mask'],
#             decoder_input['input_ids'], decoder_input['attention_mask'],
#             decoder_target
#         ):
#             yield ({
#                 "encoder_input_ids": enc_in,
#                 "encoder_attention_mask": enc_mask,
#                 "decoder_input_ids": dec_in,
#                 "decoder_attention_mask": dec_mask,
#             }, dec_tgt)

#     # Use from_generator to avoid storing the entire dataset in memory
#     dataset = tf.data.Dataset.from_generator(
#         gen,
#         output_signature=(
#             {
#                 "encoder_input_ids": tf.TensorSpec(shape=(None,), dtype=tf.int32),
#                 "encoder_attention_mask": tf.TensorSpec(shape=(None,), dtype=tf.int32),
#                 "decoder_input_ids": tf.TensorSpec(shape=(None,), dtype=tf.int32),
#                 "decoder_attention_mask": tf.TensorSpec(shape=(None,), dtype=tf.int32),
#             },
#             tf.TensorSpec(shape=(None,), dtype=tf.int32),
#         )
#     )

#     if shuffle_size:
#         dataset = dataset.shuffle(shuffle_size, reshuffle_each_iteration=True)

#     if replica_batch_size:
#         BATCH_SIZE = replica_batch_size * tpu_strategy.num_replicas_in_sync
#         dataset = dataset.batch(BATCH_SIZE, drop_remainder=drop_remainder, num_parallel_calls=tf.data.AUTOTUNE)

#     if cache:
#         dataset = dataset.cache()



#     return dataset.prefetch(tf.data.AUTOTUNE)


In [ ]:
# BOTTLE NEXK

train_set = create_dataset(
    df,
    tokenizer=tokenizer,
    replica_batch_size=64,
    drop_remainder=True, 
    shuffle_size=100
)

train_set

In [ ]:
class TransformerModel(tf.keras.Model):
    def __init__(self, encoder_embed_layer, decoder_embed_layer, positional_encoding, 
                 encoder_block, decoder_block, adaptive_softmax, max_seq_length, **kwargs):
        super().__init__(**kwargs)

        # Store layers and parameters
        self.encoder_embed_layer = encoder_embed_layer
        self.decoder_embed_layer = decoder_embed_layer
        self.positional_encoding = positional_encoding
        self.encoder_block = encoder_block
        self.decoder_block = decoder_block
        self.adaptive_softmax = adaptive_softmax
        self.max_seq_length = max_seq_length
    
    def call(self, inputs, training=False):
        encoder_input_ids = inputs["encoder_input_ids"] 
        decoder_input_ids = inputs["decoder_input_ids"]
        
        encoder_pad_mask = inputs["encoder_attention_mask"][:, tf.newaxis]
        decoder_pad_mask = inputs["decoder_attention_mask"][:, tf.newaxis]


        # Causal Mask for Decoder
        causal_mask = tf.linalg.band_part(
            tf.ones((self.max_seq_length, self.max_seq_length), tf.bool), -1, 0)
        
        attention_mask1 = causal_mask & tf.cast(decoder_pad_mask, tf.bool)
        
        # Encoder processing
        Z = self.encoder_embed_layer(encoder_input_ids)
        Z = self.positional_encoding(Z)
        Z = self.encoder_block(inputs=Z, attention_mask=encoder_pad_mask)
    
        # Decoder processing
        X = self.decoder_embed_layer(decoder_input_ids)
        X = self.positional_encoding(X)
        logits = self.decoder_block(inputs=X, encoder_output=Z,
                               attention_mask1=attention_mask1, attention_mask2=encoder_pad_mask
                              )
        return logits
        
        
    def generate(self,
                 inputs, 
                 src_tokenizer, 
                 tgt_tokenizer=None, 
                 start_token=None, 
                 end_token=None,
                 beam_width=5,
                 **kwargs):
        
        self.src_tokenizer =  src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer or src_tokenizer
        
        inputs = self.src_tokenizer.encode(inputs, **kwargs)

        Z = self.encoder_embed_layer(inputs['input_ids'])
        Z = self.positional_encoding(Z)
        encoder_output = self.encoder_block(inputs=Z, attention_mask=inputs['attention_mask'])
        
        # Initialize the beam search candidates
        start_token = start_token or self.tgt_tokenizer.encode([''],  add_bos=True, out_type=int, with_attention_mask=False)[0][0]
        end_token = start_token or self.tgt_tokenizer.encode([''],  add_eos=True, out_type=int, with_attention_mask=False)[0][0]
        
        sequences = [(tf.constant([start_token], dtype=tf.int32), 1.0)]  # (sequence, score)
        
        for _ in range(self.max_seq_length):
            all_candidates = []
            for seq, score in sequences:
                if seq[-1].numpy() == end_token:
                    all_candidates.append((seq, score))  # Keep completed sequences
                    continue
                
                # Expand sequence for decoding
                seq_padded = tf.keras.preprocessing.sequence.pad_sequences([seq.numpy()], maxlen=self.max_length, padding='post')
                decoder_mask = tf.cast(seq_padded != 0, dtype=tf.int32)
                
                # Predict next token probabilities
                Z = self.decoder_embed_layer(seq_padded)
                Z = self.positional_encoding(Z)
                decoder_output = self.decoder_block(Z, encoder_output, attention_mask1=None, attention_mask2=decoder_mask)
                prediction= self.adaptive_softmax(decoder_output)
                predictions = predictions[:, -1, :]  # Take last token probability distribution
                
                # Get top-k (beam_width) next tokens
                top_k_probs, top_k_tokens = tf.math.top_k(predictions, k=self.beam_width)
                
                for i in range(self.beam_width):
                    next_token = top_k_tokens[0][i].numpy()
                    next_score = score * top_k_probs[0][i].numpy()
                    new_seq = tf.concat([seq, [next_token]], axis=0)
                    all_candidates.append((new_seq, next_score))
                
            # Sort candidates by score and keep top-k sequences
            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
        
        # Choose the best sequence (highest score)
        best_seq = sequences[0][0].numpy()
        translated_sentence = self.tgt_tokenizer.decode([best_seq], out_type=str)
        return translated_sentence.replace('<start>', '').replace('<end>', '').strip()

    

    def get_config(self):
        """Returns a dictionary containing the model configuration"""
        config = super(TransformerModel, self).get_config()
        config.update({
            "max_seq_length": self.max_seq_length,
        })
        return config

    @classmethod
    def from_config(cls, config):
        """Recreates the model from its config"""
        return cls(**config)

In [ ]:
N = 6
n_units = 128    #First Dense layer unit in each feedforward block
num_heads = 8
proj_dims = []
proj_factor = 4
embed_size = 128
dropout_rate = 0.1
vocab_size = tokenizer.get_piece_size() #32000
batch_max_len_dec = MAX_SEQ_LENGTH 
cutoffs = [2_000, 10_000, vocab_size]

with tpu_strategy.scope():
    encoder_embed_layer =  tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embed_size,
        embeddings_initializer='glorot_uniform',
        mask_zero=True,
    )

    decoder_embed_layer =  tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embed_size,
        embeddings_initializer='glorot_uniform',
        mask_zero=True,
    )

    encoder_block = EncoderTransformerBlock(n_units=n_units,
                                          num_heads=num_heads,
                                          dropout_rate=dropout_rate,
                                          N=N,
                                        )
    
    decoder_block = DecoderTransformerBlock(n_units=n_units,
                                          num_heads=num_heads,
                                          dropout_rate=dropout_rate,
                                          N=N,
                                        )
    
    positional_encoding = PositionalEncoding()
    
    adaptive_softmax = AdaptiveSoftmax(cutoffs=cutoffs, proj_factor=proj_factor)


    model = TransformerModel(
        encoder_embed_layer=encoder_embed_layer,
        decoder_embed_layer=decoder_embed_layer,
        positional_encoding=positional_encoding, 
        encoder_block=encoder_block, 
        decoder_block=decoder_block,
        adaptive_softmax=adaptive_softmax,
        max_seq_length=MAX_SEQ_LENGTH
    )

    # one_global_batch = dataset.take(1)
    # replica_batch = one_global_batch.unbatch().batch(replica_batch_size)
    # for one_batch in replica_batch.take(1):
    #     model.build(one_batch[0].shape)

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001,
                                         clipnorm=1.0
                                        )


    model.compile(
        optimizer=optimizer,
        loss=adaptive_softmax
    )

In [ ]:
model.fit(train_set, epochs=1)

In [ ]:
# class NanInfLoggingCallback(tf.keras.callbacks.Callback):
#     def __init__(self, log_file="/kaggle/working/tpu_log.txt"):
#         super().__init__()
#         self.log_file = log_file

#     def on_train_batch_end(self, batch, logs=None):
#         has_nan = False
#         has_inf = False

#         for layer in self.model.layers:
#             weights = layer.get_weights()
#             for w in weights:
#                 if tf.math.reduce_any(tf.math.is_nan(w)).numpy():
#                     has_nan = True
#                 if tf.math.reduce_any(tf.math.is_inf(w)).numpy():
#                     has_inf = True
#                 if has_nan or has_inf:
#                     break  # Stop checking further if NaN or Inf is found

#         with open(self.log_file, "a") as f:
#             f.write(f"Batch {batch}, has_nan: {has_nan}, has_inf: {has_inf}\n")

#         if has_nan or has_inf:
#             print(f"⚠️ ⚠️ warniWarning: NaN/Inf detected in weights at batch {batch}")


# nan_callback = NanInfLoggingCallback()

In [ ]:
# class LogitsMonitoringCallback(tf.keras.callbacks.Callback):
#     def on_train_batch_end(self, batch, logs=None):
#         # Assuming your model outputs logits directly
#         logits = self.model.output
#         min_logit = tf.reduce_min(logits)
#         max_logit = tf.reduce_max(logits)
#         tf.print(f"Batch {batch}: Min Logit = {min_logit}, Max Logit = {max_logit}")


# logits_monitor_callback = LogitsMonitoringCallback()

In [ ]:
# model.generate(inputs=['Mo fẹ́ búrẹdì'], 
#                src_tokenizer=tokenizer, 
#                tgt_tokenizer=None, 
#                start_token=None, 
#                end_token=None,)

In [ ]:
# model.save('model.keras')
# model = tf.keras.models.load_model('/kaggle/working/model.keras')